<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Machine Learning with Spark MLlib: Regression

*Session 5 · Notebook 03.02 · Lecture · Student version*

## Overview

This notebook builds a full machine-learning model **at scale** with **Spark MLlib**, the machine-learning library that runs on Spark DataFrames. It is the same workflow you already know from scikit-learn (explore, prepare features, scale, split, fit, predict, evaluate, tune), expressed in Spark so it runs on data too big for one machine.

We predict **median house value** on the California housing data. Along the way we explore the data with plots, engineer features, assemble them into a feature vector, standardise, split, fit a **multiple linear regression**, evaluate it with RMSE, MAE and R-squared, then regularise it with **Ridge** and **Lasso**. Finally we put every step into one **pipeline**, and see (as notes) how it would be tuned and saved.

**The data.** The California housing dataset comes from the 1990 US census: one row per **district** (a block group, typically 600 to 3,000 people), 20,640 districts in total, with the location, the median age of the houses, the total rooms and bedrooms, the population, the number of households, the median income and the **median house value**, our target. It is a classic regression dataset: the same data ships with scikit-learn, whose page describes it in detail: [California housing dataset (scikit-learn)](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset).

## Learning Objectives

By the end of this notebook you will be able to:

- Explore a Spark DataFrame with summary statistics and plots.
- Explain and perform feature engineering, and say why MLlib needs the features in a single vector column.
- Scale features and split the data in Spark.
- Fit a multiple linear regression with `LinearRegression` and read its coefficients.
- Fit Ridge and Lasso models, and control the strength of the penalty with `regParam`.
- Evaluate a regression (RMSE, MAE, R-squared) and visualise coefficients and predictions.
- Chain the steps into a `Pipeline`, and know how it could be saved.
- Explain what `CrossValidator` would add, and why it is not run on the course compute.

## Prerequisites

- Notebook 01.02 (PySpark fundamentals: SparkSession, DataFrames, schemas).
- The Session 4 regression notebooks: the concepts (features, scaling, train/test split, RMSE/R-squared, regularisation) are identical here.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is Spark MLlib?](#sec2)
3. [Load the California housing data](#sec3)
4. [Explore the data](#sec4)
5. [Feature engineering](#sec5)
6. [Assemble features into a vector (and why)](#sec6)
7. [Standardise the features](#sec7)
8. [Train / test split](#sec8)
9. [Multiple linear regression](#sec9)
10. [Predict and evaluate (with plots)](#sec10)
11. [Ridge and Lasso: controlling the penalty](#sec11)
12. [Pipelines: all the steps in one object](#sec12)
13. [Tuning with cross-validation (a note)](#sec13)
14. [Reproducibility and saving a model (a note)](#sec14)
15. [Exercises](#exercises)
16. [Key Takeaways](#takeaways)

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Everything in the Session 4 regression and classification notebooks assumed the data fits in memory. In production, a risk model may be trained on **hundreds of millions of rows** (every transaction, every account-month). Spark MLlib lets you run the *same* modelling workflow on that scale, inside the same distributed platform that already holds the data, so you avoid sampling the data down just to fit it on one machine. The skills transfer directly: this is your scikit-learn knowledge, expressed for a cluster.

<a id="sec2"></a>
# Section 2: What is Spark MLlib?

**Definition:** MLlib is Spark's machine-learning library. It provides the familiar estimators (linear and logistic regression, decision trees, random forests, gradient-boosted trees, k-means, and more) that train on Spark DataFrames across a cluster.

**Example:** fitting a linear regression on a billion-row table: MLlib distributes the fitting across the cluster and returns one model, using the same `fit` / `transform` idea as scikit-learn.

**Analogy:** it is the scikit-learn toolbox, re-built to work in Spark's distributed kitchen instead of on a single benchtop.

**Explanation (the one big difference from scikit-learn):**

- In scikit-learn, `X` is a table of feature columns. In **MLlib, every model expects all the features packed into a single vector column**, which you build with a `VectorAssembler` (Section 6 explains why).
- MLlib estimators take `featuresCol` and `labelCol` arguments naming those columns, and, like Spark generally, transformations are **lazy** until an action runs.
- The overall recipe is: **explore, engineer features, assemble into a vector, scale, split, fit, predict, evaluate, tune.**

Two words you will see throughout: an **estimator** learns from data with `fit()` and returns a fitted **model** (for example `StandardScaler`, `LinearRegression`); a **transformer** changes a DataFrame with `transform()` (for example `VectorAssembler`, or any fitted model).

<a id="sec3"></a>
# Section 3: Load the California housing data

The file `cal_housing.data` has **no header**, so Spark cannot learn the column names from it. We define an explicit schema (column names and types) and pass it to the read. The columns are geographic and demographic features of California districts, and the target `medhv` is the median house value.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import numpy as np
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

rnd_seed = 42      # one seed for every random step (split, cross-validation folds)
from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_03_02_Regression").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# The file has no header row, so we tell Spark the column names and types up front
schema = T.StructType([
    T.StructField("long", T.DoubleType(), True),      # longitude of the district
    T.StructField("lat", T.DoubleType(), True),       # latitude of the district
    T.StructField("medage", T.DoubleType(), True),    # median house age
    T.StructField("totrooms", T.DoubleType(), True),  # total rooms in the district
    T.StructField("totbdrms", T.DoubleType(), True),  # total bedrooms in the district
    T.StructField("pop", T.DoubleType(), True),       # population
    T.StructField("houshlds", T.DoubleType(), True),  # number of households
    T.StructField("medinc", T.DoubleType(), True),    # median income
    T.StructField("medhv", T.DoubleType(), True),     # median house value  <-- the target
])

# Default: the pandas bridge. header=None tells pandas the file has no header row;
# schema= then names the columns and sets their types.
cal_housing_df = pandas_bridge(spark, "cal_housing", schema=schema, header=None)



In [ ]:
# Your turn. Write your solution here:

<a id="sec4"></a>
# Section 4: Explore the data

Before modelling, understand the data: its size, its types, its summary statistics, and the shape of the target and its relationships. In Spark we compute the summaries **on the cluster**, then pull the (small) results into pandas to plot them. Never `toPandas()` a huge DataFrame in full; summarise or sample first.

In [ ]:
# Size and schema

In [ ]:
# Summary statistics (computed in Spark, shown as a small pandas table)

In [ ]:
# Distribution of the target (median house value).
# .toPandas() is safe here because we first select a single column.

In [ ]:
medhv_pd = cal_housing_df.select("medhv").toPandas()

plt.figure(figsize=(8, 4))
plt.hist(medhv_pd["medhv"], bins=40, color="steelblue")
plt.xlabel("median house value"); plt.ylabel("count")
plt.title("Target distribution: note the spike at the top (values are capped)")
plt.show()

In [ ]:
# Relationship between median income and median house value (a strong predictor).
# We sample to keep the scatter light, then plot with pandas.

In [ ]:
sample_pd = cal_housing_df.select("medinc", "medhv").sample(fraction=0.1, seed=rnd_seed).toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(sample_pd["medinc"], sample_pd["medhv"], s=8, alpha=0.3)
plt.xlabel("median income"); plt.ylabel("median house value")
plt.title("Higher income districts tend to have higher house values")
plt.show()

In [ ]:
# How many districts share each median-house-age value? A quick grouped count.

What the exploration tells us: the target is right-skewed and **capped** at the top (a flat spike), median income is clearly related to house value (a useful feature), and the raw totals (`totrooms`, `totbdrms`, `pop`) are district-wide sums that will be more meaningful once turned into per-household ratios, which is the next step.

<a id="sec5"></a>
# Section 5: Feature engineering

**What is feature engineering?** It is creating better input columns from the raw ones so the model has more directly useful signal. Raw district totals like `totrooms` depend on how big the district is; dividing by the number of households turns them into meaningful **per-household** quantities that compare fairly across districts.

**What we will do here:**

1. **Rescale the target** `medhv` into units of 100,000, so the numbers (and later the error metrics) are easier to read.
2. **Create ratio features** that are more informative than the raw totals:
   - `rooms_per_household` = total rooms / households
   - `pop_per_household` = population / households
   - `bedrooms_per_room` = total bedrooms / total rooms
3. **Keep only** the target and the columns we will model on.

In [ ]:
# 1. Rescale the target into units of 100,000 (so 2.5 means $250,000)
cal_df = cal_housing_df.withColumn("medhv", F.col("medhv") / 100000)

# 2. Build ratio features. F.round keeps them tidy; each withColumn adds one new column.
cal_df = (cal_df
          .withColumn("rooms_per_household", F.round(F.col("totrooms") / F.col("houshlds"), 2))  # rooms per household
          .withColumn("pop_per_household",  F.round(F.col("pop")      / F.col("houshlds"), 2))  # people per household
          .withColumn("bedrooms_per_room",  F.round(F.col("totbdrms") / F.col("totrooms"), 2))) # bedroom share

# 3. Keep the target plus the features we will actually use
cal_df = cal_df.select("medhv", "pop", "houshlds", "medinc",
                       "rooms_per_household", "pop_per_household", "bedrooms_per_room")
cal_df.show(5)

<a id="sec6"></a>
# Section 6: Assemble features into a vector (and why)

**Why a single vector column?** scikit-learn accepts a table of feature columns directly. Spark MLlib does not: for speed and consistency across a cluster, **every MLlib model expects all the features packed into one column, where each row holds a vector of that row's feature values**. The `VectorAssembler` does this packing: it takes a list of columns and produces one `features` column of vectors. This is the one extra step in every MLlib pipeline compared with scikit-learn.

In [ ]:
# List the feature columns we want to combine (we leave out lat/long here)
# VectorAssembler packs those columns into a single 'features' vector column
# Each row's 'features' is now a vector of its 6 feature values

<a id="sec7"></a>
# Section 7: Standardise the features

As with distance- and penalty-based models in scikit-learn, we standardise so no feature dominates just because of its units. MLlib's `StandardScaler` is an estimator: `fit` learns each feature's mean and standard deviation, then `transform` applies them. By default it divides each feature by its standard deviation (`withStd=True`) without subtracting the mean (`withMean=False`).

> **Note - min-max scaling.** Standardising is not the only option. `MinMaxScaler` rescales each feature to a fixed range, 0 to 1 by default, with `(x - min) / (max - min)`, like scikit-learn's `MinMaxScaler`. It is used the same way (an estimator: fit on train, transform both):
>
> ```python
> from pyspark.ml.feature import MinMaxScaler
> minmax = MinMaxScaler(inputCol="features", outputCol="features_scaled")   # min=0.0, max=1.0 by default
> ```
>
> Min-max scaling keeps the shape of each feature but is sensitive to outliers (one extreme value squeezes everything else). For regularised regression, standardising is the usual choice, so we use `StandardScaler` here.

To avoid **data leakage** we only *create* the scaler here and defer fitting it until **after** the train/test split, so it learns from the training data alone (Section 8).

In [ ]:
# Create the scaler (an estimator). IMPORTANT: to avoid data leakage its mean/std must be
# LEARNED from the TRAINING data only, so we fit it AFTER the train/test split (Section 8), not on the full data.

<a id="sec8"></a>
# Section 8: Train / test split

`randomSplit` is Spark's equivalent of `train_test_split`. We pass the proportions and a seed for reproducibility. The proportions are approximate: each row is assigned at random, so 80% of 20,640 rows gives roughly, not exactly, 16,512.

In [ ]:
# Split FIRST, on the assembled (but not-yet-scaled) data
# Fit the scaler on the TRAINING data ONLY, then apply it to both sets.
# This prevents leakage: the test set's statistics never influence the scaling.

In [ ]:
# What the scaler did: the same rows before (features) and after (features_scaled).
# Each value is divided by that feature's standard deviation in the training data.

<a id="sec9"></a>
# Section 9: Multiple linear regression

**Definition:** a **multiple linear regression** predicts the target from several features at once: `medhv = b0 + b1 * x1 + b2 * x2 + ...`. Each coefficient is the effect of its feature **holding the others constant**. With no penalty, MLlib's `LinearRegression` fits it by **ordinary least squares (OLS)**: it chooses the coefficients that make the sum of the squared errors (actual minus predicted) as small as possible. It is the same model as scikit-learn's `LinearRegression`, so below we call it the OLS model.

**Example:** house value from population, households, income and the three per-household ratios.

**Analogy:** a recipe's cost. Each ingredient has its own price per unit and the total is the sum, so changing the amount of one ingredient changes the cost by its price, whatever the others are.

**Explanation:** we fit on the **standardised** features, so each coefficient is the change in house value (in 100,000s) for a one-standard-deviation change in that feature, and the coefficients can be compared with each other.

In [ ]:
# Define the model: name the (scaled) features column and the label column.
# No penalty (regParam is 0 by default), so this is ordinary least squares (OLS).
# fit() runs the (distributed) training and returns a fitted model

In [ ]:
# Collect the intercept and one coefficient per feature into a small pandas table

**Reading the table.** Median income has by far the largest coefficient: the richer the district, the higher the house values, all else equal. `pop` and `houshlds` have large coefficients of **opposite sign**: the two columns move together (big districts have both more people and more households), so the model balances one against the other. That is a sign of correlated features, which regularisation (Section 11) deals with. `rooms_per_household` and `pop_per_household` have coefficients close to zero: once the other features are known, they add little.

In [ ]:
# Plot the coefficients so their relative influence is easy to see

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(order["feature"], order["coefficient"], color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.title("Multiple linear regression coefficients (on standardised features)")
plt.xlabel("effect on median house value (green = raises, red = lowers)")
plt.tight_layout()
plt.show()

<a id="sec10"></a>
# Section 10: Predict and evaluate (with plots)

`transform` generates predictions on the test set (adding the `predmedhv` column). We evaluate with `RegressionEvaluator`, then plot predicted against actual values, the clearest visual check of a regression.

| Metric | Meaning |
|---|---|
| **RMSE** | Typical size of an error, penalising large errors more |
| **MAE** | Average absolute error |
| **R-squared** | Share of the target's variance explained (1 is perfect, 0 is no better than the mean) |

In [ ]:
# transform() = predict on the test set (adds the 'predmedhv' column)
# The error of each prediction: actual minus predicted (positive = the model guessed too low)

In [ ]:
# Evaluate with RMSE, MAE and R2 (one evaluator per metric)

pred_and_labels = predictions.select("predmedhv", "medhv")
for metric in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(predictionCol="predmedhv", labelCol="medhv", metricName=metric)
    print(f"{metric.upper()}: {evaluator.evaluate(pred_and_labels):.4f}")

In [ ]:
# Predicted vs actual: points near the diagonal are good predictions

In [ ]:
pa = pred_and_labels.sample(fraction=0.2, seed=rnd_seed).toPandas()

plt.figure(figsize=(6, 6))
plt.scatter(pa["medhv"], pa["predmedhv"], s=8, alpha=0.3)
lims = [pa[["medhv", "predmedhv"]].min().min(), pa[["medhv", "predmedhv"]].max().max()]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("actual median house value"); plt.ylabel("predicted")
plt.title("Predicted vs actual")
plt.legend()
plt.show()

The points follow the diagonal but spread widely, and the vertical band at the right is the cap at 500,001 dollars seen in Section 4: districts whose true value is above the cap all appear at 5.0. With only six features (no location), R-squared is a little above 0.5; Exercise 3 shows how much location adds.

<a id="sec11"></a>
# Section 11: Ridge and Lasso: controlling the penalty

Recall from the Session 4 Ridge and Lasso notebooks that **regularisation** adds a penalty on the size of the coefficients to curb overfitting and to stabilise correlated features. There are two classic penalties:

- **L2 (Ridge):** penalises the sum of *squared* coefficients. It shrinks all coefficients smoothly toward zero but never exactly to zero.
- **L1 (Lasso):** penalises the sum of *absolute* coefficients. It can drive some coefficients to exactly zero, performing feature selection.

MLlib has no separate `Ridge` or `Lasso` class: both are `LinearRegression` with two settings.

| Setting | What it controls | scikit-learn equivalent |
|---|---|---|
| **`regParam`** | The **strength** of the penalty: 0 means no penalty (Section 9), larger means more shrinkage | `alpha` in `Ridge(alpha=...)` and `Lasso(alpha=...)` |
| **`elasticNetParam`** | **Which** penalty: `0.0` = L2 (**Ridge**), `1.0` = L1 (**Lasso**) | the choice of class |

So **yes: `regParam` is MLlib's name for scikit-learn's `alpha`**, the number you turn up for a stronger penalty. The values are not on exactly the same scale as scikit-learn's `alpha` (the two libraries write the penalty slightly differently), so do not copy a value across: try a few and compare them on held-out data. Values between 0 and 1 for `elasticNetParam` mix the two penalties (elastic net); we keep to pure Ridge and pure Lasso here.

Because the penalty depends on the size of each coefficient, the features must be **standardised** first, which we did in Section 8.

In [ ]:
# Ridge: L2 penalty. regParam = the strength (scikit-learn's alpha), elasticNetParam = 0.0 means Ridge
# Lasso: L1 penalty. Same model class; elasticNetParam = 1.0 means Lasso


In [ ]:

# The coefficients of the three models side by side
coef_compare = pd.DataFrame({
    "OLS": ols_model.coefficients.toArray(),
    "Ridge (0.1)": ridge_model.coefficients.toArray(),
    "Lasso (0.05)": lasso_model.coefficients.toArray(),
}, index=featurecols)
coef_compare.round(3)

In [ ]:
# The same comparison as a chart: one group of bars per feature
coef_compare.plot.barh(figsize=(9, 5), color=["grey", "steelblue", "darkorange"])
plt.axvline(0, color="black", lw=0.8)
plt.title("Coefficients: OLS vs Ridge vs Lasso (standardised features)")
plt.xlabel("coefficient")
plt.tight_layout()
plt.show()

**Reading the chart.** Ridge pulls the large, opposite-signed `pop` and `houshlds` coefficients towards each other and towards zero, and keeps every feature. Lasso goes further: it sets the weakest coefficients to **exactly zero** and keeps only the features that matter most, led by median income.

In [ ]:
# How the strength changes the model: refit at several regParam values and count the features kept
rmse_eval = RegressionEvaluator(predictionCol="predmedhv", labelCol="medhv", metricName="rmse")
rows = []
for reg in [0.01, 0.05, 0.1, 0.3]:                       # the strength (alpha)
    for name, mix in [("Ridge", 0.0), ("Lasso", 1.0)]:     # the penalty type
        m = LinearRegression(featuresCol="features_scaled", labelCol="medhv", predictionCol="predmedhv",
                             regParam=reg, elasticNetParam=mix).fit(train_data)
        kept = int((np.abs(m.coefficients.toArray()) > 1e-6).sum())     # non-zero coefficients
        rows.append((name, reg, kept, round(rmse_eval.evaluate(m.transform(test_data)), 4)))
pd.DataFrame(rows, columns=["model", "regParam", "features_kept", "test_RMSE"])

As `regParam` grows, Ridge keeps all six features with ever smaller coefficients, while Lasso drops them one after another; too much penalty and both models lose accuracy. Choosing `regParam` by looking at the **test** set, as this table does, is fine for learning but optimistic in practice: the test set should be kept for the final check. Section 13 explains the proper tool, cross-validation.

In [ ]:
# Compare the three models on the same test set
evaluators = {m: RegressionEvaluator(predictionCol="predmedhv", labelCol="medhv", metricName=m) for m in ["rmse", "mae", "r2"]}
rows = []
for name, model in [("OLS", ols_model), ("Ridge (0.1)", ridge_model), ("Lasso (0.05)", lasso_model)]:
    pred = model.transform(test_data)
    rows.append([name] + [round(evaluators[m].evaluate(pred), 4) for m in ["rmse", "mae", "r2"]])
pd.DataFrame(rows, columns=["model", "RMSE", "MAE", "R2"])

<a id="sec12"></a>
# Section 12: Pipelines: all the steps in one object

So far we ran each step by hand: assemble, split, fit the scaler, transform train and test, fit the model. It works, but every step must be repeated in the right order for every new dataset, and it is easy to fit something on the wrong data by mistake.

**Definition:** a **`Pipeline`** is an ordered list of stages. `pipeline.fit(df)` runs the stages in order, fitting each estimator on the output of the previous stages, and returns a **`PipelineModel`**: one fitted object that applies every step with `transform`.

**Analogy:** a production line. You design it once, commission it on a trial batch (fit), and from then on raw material goes in one end and the finished product comes out the other (transform), always processed in exactly the same way.

**Explanation:** a pipeline gives three things:

1. **No leakage:** every statistic (the scaler's standard deviations, the coefficients) is learned from the data passed to `fit`, which is the training data only.
2. **Consistency:** test data, and next month's data, get exactly the same preparation as the training data.
3. **One object** to evaluate (and, where the workspace allows it, to save: Section 14).

In [ ]:
# Split the UN-assembled data: the pipeline does the assembling and scaling itself
train_df, test_df = cal_df.randomSplit([0.8, 0.2], seed=rnd_seed)

# The three stages, created here so the whole recipe is in one cell
assembler = VectorAssembler(inputCols=featurecols, outputCol="features")              # 1. pack the features
standard_scaler = StandardScaler(inputCol="features", outputCol="features_scaled")    # 2. standardise them
ridge = LinearRegression(featuresCol="features_scaled", labelCol="medhv",             # 3. the Ridge model
                         predictionCol="predmedhv", regParam=0.1, elasticNetParam=0.0)

## Pipeline:  


In [ ]:
# One transform call runs every step on the raw test data

The two RMSEs are identical: the seed and the data are the same, so the split puts the same rows in each part, and the pipeline repeats exactly what we did by hand, in a single `fit` and a single `transform`.

<a id="sec13"></a>
# Section 13: Tuning with cross-validation (a note)

We picked `regParam` and the penalty type by hand. As in scikit-learn, the principled way is **cross-validation**: try a grid of values, score each with k-fold cross-validation on the training data only, and keep the best. Spark provides `ParamGridBuilder` (the grid, like the `param_grid` dictionary of `GridSearchCV`) and `CrossValidator` (the search). You pass the **whole pipeline** as the estimator, so the scaler is refitted inside every fold: no leakage, even during tuning.

> **Why we do not run it here.** On the shared and serverless Databricks compute used for this course, `CrossValidator` fails with an error about `SPARKML_TEMP_DFS_PATH`: it needs a Unity Catalog Volume to store its temporary results, and learners cannot create or configure one. The code below is for reference only (it is not a code cell). On compute where you control the storage (a classic cluster, or a workspace with a Volume set up for this), it runs as shown.

```python
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# A grid of penalty strengths and penalty types (Ridge or Lasso) to try
param_grid = (ParamGridBuilder()
              .addGrid(ridge.regParam, [0.01, 0.05, 0.1])
              .addGrid(ridge.elasticNetParam, [0.0, 1.0])      # 0 = Ridge, 1 = Lasso
              .build())

# 3-fold cross-validation over the grid (3 folds x 6 combinations = 18 fits)
cross_validator = CrossValidator(estimator=pipeline,             # the whole pipeline
                                 estimatorParamMaps=param_grid,
                                 evaluator=rmse_eval,             # lowest RMSE wins
                                 numFolds=3, seed=rnd_seed)
cv_model = cross_validator.fit(train_df)

print(cv_model.avgMetrics)                     # the average RMSE of each combination
best_pipeline = cv_model.bestModel             # refitted on all of train_df with the best values
print(best_pipeline.stages[-1].getRegParam(), best_pipeline.stages[-1].getElasticNetParam())
```

Without `CrossValidator`, a simple and honest alternative is a **validation split**: split the training data again (for example 80/20), compare a few `regParam` values on that validation part, and keep the test set untouched for the final score.

<a id="sec14"></a>
# Section 14: Reproducibility and saving a model (a note)

A model is **reproducible** when the same code on the same data gives the same model. The habits that make it so:

| Source of change | Control |
|---|---|
| `randomSplit`, random forests | Fix `seed` (we use `rnd_seed` everywhere) |
| Column types | Read with an explicit schema (Section 3) |
| Preparation steps | Keep them **inside** the pipeline, so one object holds every step |
| Software versions | Record `spark.version` with the model |

> **Note: saving a model.** A fitted `PipelineModel` **can** be saved and loaded back later, like `joblib.dump` and `joblib.load` in scikit-learn. The saved folder contains every stage (the assembler's column list, the scaler's standard deviations, the model's coefficients). Whether it works, and **where** you can save it, depends on how the Databricks workspace is configured: on shared and serverless compute you usually need a Unity Catalog Volume you are allowed to write to, for example `/Volumes/<catalog>/<schema>/<volume>/models/`. Ask your workspace administrator before relying on it. We do not run it in this course; the code would look like this:
>
> ```python
> from pyspark.ml import PipelineModel
>
> path = "/Volumes/<catalog>/<schema>/<volume>/models/cal_housing_ridge"
> pipeline_model.write().overwrite().save(path)     # save the fitted pipeline
> reloaded = PipelineModel.load(path)               # load it back later
> reloaded.transform(new_df)                        # raw columns in, predictions out
> ```

<a id="exercises"></a>
# Exercises

### Exercise 1: A simple linear regression

Fit a regression of `medhv` on `medinc` **only**, as a pipeline of a `VectorAssembler` and a `LinearRegression` (no scaler, no penalty), on `train_df`. Print the equation and the test R-squared. How does it compare with the six-feature model of Section 10?

> *Hint:* Even one feature must go into a vector, so the assembler is still needed.

In [ ]:
# Your turn

### Exercise 2: Where does Lasso start dropping features?

Fit Lasso models (`elasticNetParam=1.0`) on `train_data` with `regParam` in `[0.005, 0.02, 0.05, 0.1, 0.2]`. For each, print which features it keeps (non-zero coefficients).

In [ ]:
# Your turn

### Exercise 3: Add location to the pipeline

Location matters for house prices. Rebuild the modelling data from `cal_housing_df` with the same feature engineering as Section 5 **plus** `lat` and `long`, split it with `rnd_seed`, and fit a pipeline (assembler, scaler, Ridge with `regParam=0.01`). Compare its test RMSE and R-squared with the pipeline of Section 12.

> *Hint:* Copy the Section 5 cell, add `"lat", "long"` to the final `select`, and reuse the Section 12 pipeline pattern.

In [ ]:
# Your turn

<a id="takeaways"></a>
## Key Takeaways

| scikit-learn | Spark MLlib equivalent |
|---|---|
| `X` as many columns | one `features` vector column via `VectorAssembler` |
| `StandardScaler().fit_transform(X)` | `StandardScaler(...).fit(df).transform(df)` |
| `MinMaxScaler()` | `MinMaxScaler(...)`, used the same way |
| `train_test_split` | `df.randomSplit([0.8, 0.2], seed=...)` |
| `LinearRegression().fit(X, y)` | `LinearRegression(featuresCol=..., labelCol=...).fit(df)` |
| `Ridge(alpha=a)` | `LinearRegression(regParam=a, elasticNetParam=0.0)` |
| `Lasso(alpha=a)` | `LinearRegression(regParam=a, elasticNetParam=1.0)` |
| `model.predict(X_test)` | `model.transform(test_df)` (adds a prediction column) |
| `r2_score`, RMSE | `RegressionEvaluator(metricName="r2" / "rmse" / "mae")` |
| `Pipeline([...])` | `Pipeline(stages=[...])`; `fit` returns a `PipelineModel` |
| `GridSearchCV` | `ParamGridBuilder` + `CrossValidator` (pass the whole pipeline; not run on the course compute, see Section 13) |
| `joblib.dump` / `joblib.load` | `model.write().overwrite().save(path)` / `PipelineModel.load(path)` (depends on the workspace, see Section 14) |

**The one-line lesson:** Spark MLlib is scikit-learn's workflow for the cluster. The habits to remember are that **every model consumes a single assembled feature vector column**, that `regParam` is the penalty strength and `elasticNetParam` chooses Ridge or Lasso, and that a **pipeline** keeps preparation and model together so they are fitted as one.

## Conclusion

You can now run an end-to-end regression at scale with Spark MLlib: explore with plots, engineer features, assemble and scale them, split, fit a multiple linear regression and read its coefficients, regularise it with Ridge and Lasso, evaluate and visualise the results, then chain everything into a pipeline. You also know how `CrossValidator` would tune it and how a pipeline would be saved, where the workspace allows. The practical (03.03) has you build the pipeline yourself; notebook 03.04 applies the same pattern to classification with decision trees, random forests and logistic regression.

On a laptop, release the resources when you finish with `spark.stop()`. **Do not** do this on Databricks, where the platform manages the session.

In [ ]:
# spark.stop()   # local environments only; never on Databricks

<a id="reading"></a>
## Further Reading & Resources

- [Spark MLlib guide](https://spark.apache.org/docs/latest/ml-guide.html)
- [`VectorAssembler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html), [`LinearRegression`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html), [`RegressionEvaluator`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)
- [ML Pipelines (including saving and loading)](https://spark.apache.org/docs/latest/ml-pipeline.html)
- [`CrossValidator` and tuning](https://spark.apache.org/docs/latest/ml-tuning.html)